<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c06-header.png" alt="Nextia Learning · Deep Learning and Transformers Explained" width="100%">

# Save and inspect results

**[Save and inspect results](https://learning.nextia-ai.com/courses/deep-learning/m07/save-and-inspect-results/)** · Deep Learning and Transformers Explained · Module 7, lesson 3 of 3 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**You will learn to** save a trained network as a `state_dict` checkpoint, reload it in a fresh process and check that its predictions are the same, see why a checkpoint alone is not enough, and write an experiment record that another person can repeat. The module practice runs SmolLM2-135M under controlled settings.

| | |
|---|---|
| **Time** | About 40 minutes, with the module practice |
| **Needs** | An internet connection for the setup cells (2 MB for the tickets, about 270 MB for the model). No account, no graphics card. About 600 MB of free memory. |
| **You should know** | The training loop with validation, from [Batches, epochs and validation](https://learning.nextia-ai.com/courses/deep-learning/m03/batches-epochs-and-validation/), and SmolLM2's generation settings, from [Training versus inference](https://learning.nextia-ai.com/courses/deep-learning/m07/training-versus-inference/). |
| **Tested** | Python 3.12 with PyTorch 2.14.1 (CPU), in Jupyter on macOS, from a fresh start (*Restart and run all*); Colab and Kaggle have PyTorch preinstalled; Hugging Face transformers 5.19.0 |

### How to use this notebook

This notebook has every query, task and check of the lesson. The [lesson page](https://learning.nextia-ai.com/courses/deep-learning/m07/save-and-inspect-results/) has the full explanations, the diagrams and the knowledge checks that count toward your certificate. Keep both open.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, write your answer in the cell, run it, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/usmanahmed99/nextia-learning-labs/blob/main/C06/M07-L03-save-and-inspect-results/save-and-inspect-results-solution.ipynb).

## Setup: check PyTorch

Run the next cell. It uses the PyTorch that is already installed (Colab and Kaggle have it) or installs the CPU version if it is missing; that can take two minutes the first time. You should see a line that starts with `PyTorch 2.` and the device `cpu`. The course uses the CPU only, so a graphics card is not needed. It also installs Hugging Face `transformers` for the small pretrained model SmolLM2-135M (about 270 MB to download later).

In [ ]:
try:
    import torch
except ImportError:  # not installed: install the CPU version
    %pip install -q --disable-pip-version-check torch --index-url https://download.pytorch.org/whl/cpu
    import torch
%pip install -q --disable-pip-version-check "transformers>=5,<6" safetensors
import transformers; print("transformers", transformers.__version__)
import platform
torch.manual_seed(0)
print("PyTorch", torch.__version__, "| Python", platform.python_version(), "| device cpu")

## Setup: get the tickets and the course's code

Run the next cell. It downloads Larkfield's ticket texts (about 2 MB) and the course's `ticketnet.py` into a folder `ticket-router`, and checks each file's checksum. It needs an internet connection and no account, and takes about 10 seconds.

You should see `Ready: train 4696 tickets, valid 600, test 600.` If you run the cell again, it uses the files that are already there.

If the cell shows an error, read its last line:

- `Could not download`: check your internet connection, then run the cell again. In Kaggle, turn on **Internet** in the notebook settings.
- `wrong checksum`: delete the folder `ticket-router` (in Colab: the **Files** panel on the left), then run the cell again.

The tickets are made up for this course by a language model, from a list of situations; no real customer wrote them.

In [ ]:
# Setup: download the tickets and ticketnet.py into ticket-router/ and check them.
import hashlib, os, subprocess, sys, urllib.request
from pathlib import Path

LABS = os.environ.get("NX_LABS_RAW", "https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/C06/")
FILES = {
    "train.csv": "98487169346f8b0d9bc55a3844849ab95d012b75f01d7083605a6a1cea57aa39",
    "valid.csv": "f0c7bfa75e75837cdaf891ac8d007a78a018ba8e3488155cf43f2c6a34a064c8",
    "test.csv": "f373f402bcc9742cdcf8829720710a97b7a1dce5d8e34aaa88d03e7e49a5404e",
    "split_manifest.json": "d043cf720697054fb16e6cfff72ecb49b471d75038308ad5836779a0dd0bf8a5",
}
if Path(".nextia-notebook").exists():  # the cell ran before: start from the parent folder
    os.chdir("..")
PROJECT = Path("ticket-router").resolve()
(PROJECT / "data").mkdir(parents=True, exist_ok=True)
(PROJECT / ".nextia-notebook").touch()

def download(url, path):
    try:
        urllib.request.urlretrieve(url, path)
    except OSError:  # Python without certificates (python.org on macOS): try curl
        if subprocess.run(["curl", "-fsSL", "-o", str(path), url]).returncode:
            raise SystemExit(f"Could not download {url}. Check your internet connection.")

for name, expected in FILES.items():
    path = PROJECT / "data" / name
    if not path.exists() or hashlib.sha256(path.read_bytes()).hexdigest() != expected:
        download(LABS + "data/" + name, path)
        if hashlib.sha256(path.read_bytes()).hexdigest() != expected:
            raise SystemExit(f"{name} has the wrong checksum. Delete the folder ticket-router and run again.")
download(LABS + "project/ticketnet.py", PROJECT / "ticketnet.py")
os.chdir(PROJECT)
sys.path.insert(0, str(PROJECT))
import ticketnet
from ticketnet import *  # TEAMS, Vocab, tokenize, the three networks, train(), evaluate() ...
train_rows, valid_rows, test_rows = (load_split(f"data/{p}.csv") for p in ("train", "valid", "test"))
print(f"Ready: train {len(train_rows)} tickets, valid {len(valid_rows)}, test {len(test_rows)}.")

## Setup: helpers

The next cell defines the check helpers. They tell you if your answer is right, without showing the answer:

- `expect(what, yours, expected)` compares one of your numbers with the expected one, and says "too high" or "too low".
- `expect_hash(what, yours, expected)` compares any result with a hidden fingerprint of the expected one.
- `show_shape(name, tensor)` prints a tensor's shape, as the lessons do.

Run the cell. It prints nothing.

In [ ]:
# Helpers: check an answer without showing it.
import hashlib, json, warnings
import numpy as np
import pandas as pd
pd.set_option("display.precision", 4)
pd.set_option("display.max_columns", 30)
warnings.filterwarnings("ignore", category=FutureWarning)

def fingerprint(value):
    def norm(v):
        if hasattr(v, "to_dict") and hasattr(v, "columns"):
            return {"columns": [str(c) for c in v.columns], "rows": [norm(list(r)) for r in v.itertuples(index=False)]}
        if hasattr(v, "tolist"):
            return norm(v.tolist())
        if isinstance(v, dict):
            return {str(k): norm(x) for k, x in sorted(v.items(), key=lambda kv: str(kv[0]))}
        if isinstance(v, (list, tuple)):
            return [norm(x) for x in v]
        if isinstance(v, bool) or v is None or isinstance(v, int):
            return v
        if isinstance(v, float):
            return None if v != v else round(v, 4)
        if hasattr(v, "item"):
            return norm(v.item())
        return str(v)
    return hashlib.sha256(json.dumps(norm(value), sort_keys=True).encode()).hexdigest()[:16]

def expect(what, yours, expected, tolerance=1e-6):
    """Compare one number or value with the expected one."""
    if isinstance(yours, (np.integer, np.floating)):
        yours = yours.item()
    if isinstance(expected, float) and isinstance(yours, (int, float)):
        same = abs(yours - expected) <= tolerance * max(1, abs(expected))
    else:
        same = yours == expected
    if same:
        print(f"Check passed: {what} is {yours}.")
    elif isinstance(expected, (int, float)) and isinstance(yours, (int, float)):
        print(f"Not yet: {what} is {yours}, which is too {'high' if yours > expected else 'low'}.")
    else:
        print(f"Not yet: {what} is {yours!r}, which is not the expected value.")

def expect_hash(what, yours, expected):
    """Compare any result with the fingerprint of the expected one."""
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
    elif fingerprint(yours) == expected:
        print(f"Check passed: {what} is right.")
    else:
        print(f"Not yet: {what} is not the expected result. Read the task again, and check each step.")

torch.set_num_threads(2)  # the numbers in the lessons were measured on a CPU

def fingerprint_tensor(t):
    return fingerprint([round(float(v), 4) for v in t.flatten().tolist()])

def show_shape(name, t):
    print(f"{name:<12} shape {tuple(t.shape)}")

## Setup: the tickets as numbers

The next cell turns the tickets into word counts, as in [The forward pass](https://learning.nextia-ai.com/courses/deep-learning/m02/the-forward-pass/): a vocabulary from the training tickets, then one row of counts per ticket. It also makes `loss_fn`, the loss of this module. Run it.

> **Check.** You should see `x_train shape (4696, 2864)`, `x_valid shape (600, 2864)` and `vocabulary: 2864`.

In [ ]:
import torch
from torch import nn

# The tickets as numbers: a vocabulary from the training tickets, then word counts.
vocab = Vocab([r["text"] for r in train_rows])
x_train, y_train = bag_of_words(train_rows, vocab), labels(train_rows)
x_valid, y_valid = bag_of_words(valid_rows, vocab), labels(valid_rows)
loss_fn = nn.CrossEntropyLoss()
show_shape("x_train", x_train)
show_shape("x_valid", x_valid)
print("vocabulary:", len(vocab), "| teams:", TEAMS)

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. Train the router, then save its weights

The next cell trains the bag-of-words router for 12 epochs, the best epoch of the course's run. It takes about 5 seconds.

In [ ]:
set_seed(0)
model = BagOfWordsNet(len(vocab))
history = train(model, x_train, y_train, x_valid, y_valid, epochs=12, log=None)
print(history[-1])
model.eval()
with torch.no_grad():
    valid_probs = torch.softmax(model(x_valid), dim=1)   # keep the predictions, to compare later

> **Check.** You should see epoch 12 with `'valid_loss': 0.0633` and `'valid_acc': 0.9833`.

A **checkpoint** is a saved copy of a model's learned numbers. `model.state_dict()` returns them as a dictionary: the name of each weight tensor and the tensor itself.

> **Predict.** The network is 2,864 → 64 → 5. How many tensors will the `state_dict` have, and which shapes?

In [ ]:
for name, tensor in model.state_dict().items():
    print(f"{name:16} {tuple(tensor.shape)}")

> **Check.** You should see four tensors: `layers.0.weight (64, 2864)`, `layers.0.bias (64,)`, `layers.3.weight (5, 64)` and `layers.3.bias (5,)`. ReLU and dropout (positions 1 and 2) have no weights.

The weights alone cannot route a ticket. The network also needs the **vocabulary**: which column of the input means which word. And whoever rebuilds the network needs its **configuration**: the sizes and settings. The next cell saves all three into a folder `checkpoint`.

In [ ]:
import json
from pathlib import Path

Path("checkpoint").mkdir(exist_ok=True)
torch.save(model.state_dict(), "checkpoint/router.pt")                     # the weights
Path("checkpoint/vocab.json").write_text(json.dumps(vocab.itos))           # the tokens, in column order
Path("checkpoint/config.json").write_text(json.dumps({"vocab_size": len(vocab), "hidden": 64, "dropout": 0.0}))
torch.save(valid_probs, "checkpoint/valid_probs.pt")                       # only for the comparison below
for f in sorted(Path("checkpoint").iterdir()):
    print(f"{f.name:16} {f.stat().st_size:>9,} bytes")

> **Check.** You should see `router.pt` with about 737,000 bytes and `vocab.json` with about 28,000 bytes.

## 2. Reload in a fresh process

A notebook remembers everything you ran. To prove that the files are enough, load them in a **new Python process** that knows nothing about this notebook. The next cell writes a small script; the cell after it runs the script with a new Python.

In [ ]:
%%writefile reload_router.py
# A fresh Python process: rebuild the network, load the saved weights, predict, compare.
import json, torch
from ticketnet import BagOfWordsNet, Vocab, bag_of_words, labels, load_split

config = json.load(open("checkpoint/config.json"))
vocab = Vocab([])                                        # an empty vocabulary ...
vocab.itos = json.load(open("checkpoint/vocab.json"))    # ... filled with the saved tokens, in order
vocab.stoi = {token: i for i, token in enumerate(vocab.itos)}

model = BagOfWordsNet(config["vocab_size"], hidden=config["hidden"], dropout=config["dropout"])
model.load_state_dict(torch.load("checkpoint/router.pt"))
model.eval()                                             # inference: dropout off

valid = load_split("data/valid.csv")
with torch.no_grad():
    probs = torch.softmax(model(bag_of_words(valid, vocab)), dim=1)
before = torch.load("checkpoint/valid_probs.pt")
print("same predictions:", (probs.argmax(1) == before.argmax(1)).sum().item(), "of", len(valid))
print("largest difference in a probability:", (probs - before).abs().max().item())
print("validation accuracy:", round((probs.argmax(1) == labels(valid)).float().mean().item(), 4))

> **Predict.** Will the reloaded network give exactly the same probabilities?

In [ ]:
import subprocess, sys
result = subprocess.run([sys.executable, "reload_router.py"], capture_output=True, text=True)
print(result.stdout or result.stderr)

> **Check.** You should see `same predictions: 600 of 600`, `largest difference in a probability: 0.0` and `validation accuracy: 0.9833`.

Note `model.eval()` in the script. This network has no active dropout, but a network with dropout would give changing answers without it, as in Module 3 and 4.

## 3. Why not save the whole object?

`torch.save(model)` also works: it saves the whole Python object. But the file then holds a **reference to the class** `BagOfWordsNet` in `ticketnet.py`, not its code. The next cell saves the object, then tries to load it in a fresh process twice: with PyTorch's safe default, and in a folder without `ticketnet.py`.

In [ ]:
import re, tempfile
torch.save(model, "checkpoint/router-object.pt")
def clean(text):  # the last line that names an error, without terminal colour codes
    lines = [re.sub(r"\x1b\[[0-9;]*m", "", l) for l in text.splitlines()]
    return [l for l in lines if re.match(r"^[\w.]+(Error|Exception)\b", l)][-1]

try_load = "import sys, torch; torch.load(sys.argv[1]{extra})"
path = str(Path("checkpoint/router-object.pt").resolve())
safe = subprocess.run([sys.executable, "-c", try_load.format(extra=""), path], capture_output=True, text=True)
print("default load:      ", clean(safe.stderr)[:60], "…")
elsewhere = subprocess.run([sys.executable, "-c", try_load.format(extra=", weights_only=False"), path],
                           capture_output=True, text=True, cwd=tempfile.gettempdir())
print("without ticketnet: ", clean(elsewhere.stderr))

> **Check.** The first line starts with `_pickle.UnpicklingError: Weights only load failed`. The second line is `ModuleNotFoundError: No module named 'ticketnet'`.

The safe default refuses to rebuild Python objects, because loading an object can run code from the file. With the safety switched off, the object still needs the exact class code, in the same place. If someone renames the file or changes the class, old checkpoints stop loading. **Save the `state_dict`, and keep the code that builds the network.**

## 4. Failure case: the wrong vocabulary

Tomás sends Amira only `router.pt`. She builds the vocabulary herself, from the training **and** validation tickets.

> **Predict.** Will the weights load into her network?

In [ ]:
amira_vocab = Vocab([r["text"] for r in train_rows + valid_rows])
amira_model = BagOfWordsNet(len(amira_vocab))
try:
    amira_model.load_state_dict(torch.load("checkpoint/router.pt"))
except RuntimeError as error:
    print(str(error).splitlines()[-1].strip())

> **Check.** You should see `size mismatch for layers.0.weight: copying a param with shape torch.Size([64, 2864]) from checkpoint, the shape in current model is torch.Size([64, 2987]).`

That error is the lucky case. If her vocabulary had the same size but another order, every word would land in another word's column, and nothing would complain. The next cell makes that happen on purpose: the same 2,864 tokens in reverse order.

In [ ]:
reversed_vocab = Vocab([])
reversed_vocab.itos = vocab.itos[:2] + vocab.itos[2:][::-1]    # keep <pad> and <unk>, reverse the rest
reversed_vocab.stoi = {token: i for i, token in enumerate(reversed_vocab.itos)}
model.eval()
with torch.no_grad():
    wrong = model(bag_of_words(valid_rows, reversed_vocab)).argmax(1)
print("accuracy with the reversed vocabulary:", round((wrong == y_valid).float().mean().item(), 4))

> **Check.** You should see `accuracy with the reversed vocabulary: 0.24`: no error, and worse than always saying "delivery". **Save the vocabulary with the weights**, and check the predictions after every reload.

## 5. Record what else matters

Weights, vocabulary and configuration rebuild the model. To repeat the **experiment**, another person also needs the seeds, the library versions, the hardware and the exact data. The next cell collects them and checks the data files against the checksums that the setup cell used.

In [ ]:
import hashlib, platform
print("Python", platform.python_version(), "| PyTorch", torch.__version__, "| machine", platform.machine(), platform.system())
print("threads:", torch.get_num_threads())
for name in ("train.csv", "valid.csv"):
    digest = hashlib.sha256(Path("data", name).read_bytes()).hexdigest()
    print(f"{name:10} {digest[:16]}…  matches SHA256SUMS: {digest == FILES[name]}")

> **Check.** You should see your versions and `matches SHA256SUMS: True` for both files. The checksum of `train.csv` starts with `98487169346f8b0d`.

## 6. Module practice: SmolLM2 under controlled settings

Now run SmolLM2-135M as an experiment that someone else can repeat. The next cells load the model; then you compare its outputs under controlled settings and write an `EXPERIMENT.md`.

## Setup: load SmolLM2-135M

Run the next cell. It downloads the small pretrained model **SmolLM2-135M** from Hugging Face and loads it. The model is free and open (licence Apache-2.0), and it needs no account.

- The first run downloads about **270 MB**. The makers store the weights in the 16-bit format bfloat16, 2 bytes per weight. The download can take a minute; later runs use the saved copy.
- The cell loads the weights as **float32**, 4 bytes per weight, so the model takes about **540 MB** of memory. The lessons' numbers were measured in float32, and every CPU computes float32 well. [Inference costs](https://learning.nextia-ai.com/courses/deep-learning/m07/inference-costs/) explains the difference.
- `revision=` asks for one exact version of the model files, so your model is the course's model, even if the makers update theirs.

You should see a progress bar, then `parameters: 134515008`.

If the cell shows an error:

- `OSError` or `Connection`: check your internet connection, then run the cell again. In Kaggle, turn on **Internet** in the notebook settings.
- `No module named 'transformers'`: run the first setup cell again.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL = "HuggingFaceTB/SmolLM2-135M"
REVISION = "93efa2f097d58c2a74874c7e644dbc9b0cee75a2"  # the exact version of the files that the course used
tokenizer = AutoTokenizer.from_pretrained(MODEL, revision=REVISION)
model = AutoModelForCausalLM.from_pretrained(MODEL, revision=REVISION, dtype=torch.float32)
model.eval()                                   # inference: switch off training-only behaviour
PROMPT = ("Larkfield is an online shop for home and garden products. "
          "A customer wrote: my parcel is late. The support team")
print("parameters:", sum(p.numel() for p in model.parameters()))

> **Your turn.** In the next cell:

1. Generate 40 greedy tokens twice, and put `True` in `greedy_same` if the two texts are identical.
2. Generate 25 tokens at temperature 0.8 (`top_p=0.95`) with seed 1, twice. Put the result of the comparison in `seed_same`.
3. For each temperature 0.3, 0.8 and 1.2, generate 25 tokens with seeds 1, 2 and 3. Put the texts in `by_temperature`, a dictionary `{temperature: [three texts]}`.

The helper `continue_text(**settings)` returns the new text for one setting. It takes about 20 seconds in all.

In [ ]:
inputs = tokenizer(PROMPT, return_tensors="pt")

def continue_text(seed=0, **settings):
    """The new text after the prompt, for one setting. The seed fixes the random draws."""
    torch.manual_seed(seed)
    with torch.no_grad():
        out = model.generate(**inputs, pad_token_id=tokenizer.eos_token_id, **settings)
    return tokenizer.decode(out[0, inputs["input_ids"].shape[1]:])

greedy_same = ...      # 1. greedy, 40 tokens, twice: identical?
seed_same = ...        # 2. seed 1, temperature 0.8, top_p 0.95, 25 tokens, twice: identical?
by_temperature = ...   # 3. {0.3: [3 texts], 0.8: [3 texts], 1.2: [3 texts]}

In [ ]:
if any(v is Ellipsis for v in (greedy_same, seed_same, by_temperature)):
    print("Not yet: write your answers in the cell above, then run it again.")
elif not (greedy_same is True and seed_same is True):
    print("Not yet: with the same settings and the same seed, the two texts should be identical. Check the seed and the settings.")
elif sorted(by_temperature) != [0.3, 0.8, 1.2] or any(len(v) != 3 for v in by_temperature.values()):
    print("Not yet: by_temperature needs the keys 0.3, 0.8 and 1.2, each with 3 texts.")
elif len(set(by_temperature[0.8])) != 3:
    print("Not yet: the three seeds at temperature 0.8 should give three different texts. Use seeds 1, 2 and 3.")
else:
    print("Check passed: the same settings repeat, and the seeds change the samples.")

> **Check.** On the course's computer, seed 2 at temperature 0.3 gives `' is very helpful. I am looking for a new home. …'`, close to the greedy text, and seed 3 at 1.2 starts `' took my item and returned it to me'`.

> **Your turn.** Fill in the four `...` lines of the record below, then run the cell. It writes `EXPERIMENT.md` next to the notebook. The lesson page has a model answer.

In [ ]:
record = f"""# Experiment: SmolLM2-135M continuations under controlled settings

## Question
...  (one sentence: what did you compare?)

## Model and code
- Model: {MODEL}, revision {REVISION}, loaded as float32
- Libraries: Python {platform.python_version()}, PyTorch {torch.__version__}, transformers {transformers.__version__}

## Settings
- Prompt: {PROMPT!r}
- ...  (greedy: how many tokens; sampling: temperatures, top_p, seeds, tokens)

## Hardware
- {platform.machine()}, {platform.system()}, CPU only, {torch.get_num_threads()} threads

## Results
- Greedy, run twice: identical = {greedy_same}
- Seed 1 at temperature 0.8, run twice: identical = {seed_same}
- ...  (one line per temperature: what the three samples did)

## Limits
- ...  (hardware, data, and what this experiment cannot tell)
"""
Path("EXPERIMENT.md").write_text(record)
print(record)

In [ ]:
text = Path("EXPERIMENT.md").read_text() if Path("EXPERIMENT.md").exists() else ""
missing = [h for h in ("## Question", "## Model and code", "## Settings", "## Hardware", "## Results", "## Limits") if h not in text]
if not text:
    print("Not yet: EXPERIMENT.md does not exist. Run the cell above.")
elif missing:
    print("Not yet: these headings are missing:", missing)
elif "..." in text:
    print("Not yet: the record still has a '...' line. Replace each one with your own words.")
else:
    print("Check passed: EXPERIMENT.md has every section.")

## Next

You can now save a network as a `state_dict` with its vocabulary and configuration, prove the reload in a fresh process, and write an experiment record with its settings, versions, hardware, data checksums and limits. The [lesson page](https://learning.nextia-ai.com/courses/deep-learning/m07/save-and-inspect-results/) has the module check, including the completion check: deliver a reproducible experiment and state its hardware and data limits. Next come the case studies, starting with [Classify clothing images](https://learning.nextia-ai.com/courses/deep-learning/m09/classify-clothing-images/).

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [Classify clothing images](https://learning.nextia-ai.com/courses/deep-learning/m09/classify-clothing-images/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/deep-learning/m07/save-and-inspect-results/)

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). The tickets are made up for this course by a language model; no real customer wrote them. SmolLM2-135M is by Hugging Face, under the Apache-2.0 licence.